# OpenPI Training

Notebook untuk melatih model VLA (Vision Language Action) Pi0 dengan 12 langkah pipeline komprehensif.

## Pipeline (12 Langkah):
1. **Setup Environment** - Deteksi dataset Kaggle
2. **Install Dependencies** - PyTorch, JAX, Transformers, dll
3. **Setup Data** - Copy/extract dataset dan normalization stats
4. **Patch Libraries** - Fix kompatibilitas (OpenPI, LeRobot, Transformers)
5. **Configure Training** - Set hyperparameter training
6. **Pre-download & LoRA** - Download tokenizer dan install PEFT
7. **Inject LoRA** - Setup low-rank adaptation pada script
8. **Resume Training** (OPSIONAL) - Restore checkpoint jika ada
9. **Launch Training** - Main training loop
10. **Monitor** - Track checkpoint progress
11. **Download** - Compress dan download model
12. **Test Inference** (OPSIONAL) - Test pada sample image

**PENTING:** Restart kernel setelah STEP 2 (Installation) sebelum melanjutkan

# STEP 1: Setup Environment & Verifikasi Dataset

Deteksi keberadaan dataset Kaggle dan membaca path-path penting (OpenPI, dataset, normalization stats) yang akan digunakan pada training.

In [ ]:
import os, sys, subprocess, shutil, glob, zipfile, re, signal, time, urllib.request
import torch, lerobot, transformers
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from safetensors.torch import load_file, save_file
from datasets import load_dataset
from openpi.policies import policy_factory
from IPython.display import FileLink, display

DATASET_NAME = 'dronepivla'
KAGGLE_BASE = f'/kaggle/input/datasets/sjankaczar/{DATASET_NAME}'

paths = {
    'openpi': f'{KAGGLE_BASE}/openpi/openpi',
    'dataset': f'{KAGGLE_BASE}/drone_roblox/drone_roblox',
    'normstats': f'{KAGGLE_BASE}/norm_stats',
}

found = {name: dir_path for name, dir_path in paths.items()}
for name, dir_path in found.items():
    print(f"{name:15} {dir_path}")

FINAL_OPENPI = found.get('openpi')
FINAL_DATASET = found.get('dataset')
FINAL_NORMSTATS = found.get('normstats')

print("Setup environment complete")

# STEP 2: Install Dependencies

Menginstal semua dependencies yang diperlukan: PyTorch, JAX, Transformers, LeRobot, dan utilities lainnya. **Restart kernel setelah langkah ini sebelum melanjutkan ke step berikutnya.**

In [ ]:
%%time

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-warn-conflicts', *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0

print("Cleaning old libraries...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 
    'jax', 'jaxlib', 'numpy', 'torch', 'torchaudio', 'torchvision', '-y'],
    capture_output=True)

packages = {
    'Core': ['numpy==1.26.4'],
    'PyTorch': ['torch==2.4.0', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121',
                'safetensors>=0.4.0', 'einops>=0.8.0', 'wandb>=0.19.1'],
    'JAX': ['jax[cuda12]==0.5.3', 'jaxlib==0.5.3', 'flax==0.10.2', 'orbax-checkpoint==0.11.13', 'chex', 
            'equinox>=0.11.8', 'ml-collections==1.0.0', 'jaxtyping==0.2.36', 'ml-dtypes==0.4.1'],
    'Data': ['lerobot@git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5',
             'datasets', 'huggingface-hub'],
    'Utilities': ['tyro>=0.9.5', 'beartype==0.19.0', 'numpydantic>=1.6.6', 'sentencepiece>=0.2.0', 
                  'tqdm-loggable>=0.2', 'humanize', 'filelock>=3.16.1', 'treescope>=0.1.7', 'augmax>=0.3.4',
                  'dm-tree>=0.1.8', 'flatbuffers>=24.3.25', 'fsspec==2023.10.0', 'gcsfs', 'imageio>=2.36.1', 
                  'opencv-python-headless', 'pillow>=11.0.0', 'rich>=14.0.0', 'polars>=1.30.0',
                  'transformers==4.53.2', 'gym-aloha>=0.1.1', 'peft']
}

for category, pkgs in packages.items():
    print(f"Installing {category}...")
    pip_install(*pkgs)

print("\nInstallation Complete!")
print("Restart kernel sebelum lanjut ke step berikutnya")

# STEP 3: Setup Data & Copy Artifacts

Menyiapkan direktori data untuk LeRobot, normalization stats, dan setup OpenPI repository untuk digunakan dalam training.

In [ ]:
HOME_DIR = os.path.expanduser("~")
TARGET_DATA_DIR = os.path.join(HOME_DIR, ".cache/huggingface/lerobot/rafli/drone_roblox")

def copy_or_extract(source, target_parent, target_name):
    target_path = os.path.join(target_parent, target_name)
    if os.path.exists(target_path): shutil.rmtree(target_path)
    
    if os.path.isdir(source):
        shutil.copytree(source, target_path)
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source, 'r') as z: z.extractall(target_parent)
    
    return target_path

def find_and_move_data(source_path, pattern, target_dir):
    matches = glob.glob(os.path.join(source_path, f"**/{pattern}"), recursive=True)
    if matches:
        actual_root = os.path.dirname(os.path.dirname(matches[0]))
        os.makedirs(os.path.dirname(target_dir), exist_ok=True)
        if os.path.exists(target_dir): shutil.rmtree(target_dir)
        shutil.move(actual_root, target_dir)
        return True
    return False

print("Setting up OpenPI...")
WORK_DIR = copy_or_extract(FINAL_OPENPI, '/kaggle/working', 'openpi')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', WORK_DIR, '--no-deps'],
               capture_output=True)
print("OpenPI ready")

print("Setting up LeRobot Dataset...")
temp_data = copy_or_extract(FINAL_DATASET, '/kaggle/working', 'temp_dataset')
if find_and_move_data(temp_data, 'meta/info.json', TARGET_DATA_DIR):
    print(f"Dataset at {TARGET_DATA_DIR}")

print("Setting up Normalization Stats...")
FINAL_TARGET_NS = '/kaggle/working/assets/pi0_drone_lite/rafli/drone_roblox'
temp_ns = copy_or_extract(FINAL_NORMSTATS, '/kaggle/working', 'temp_ns')
if find_and_move_data(temp_ns, 'norm_stats.json', FINAL_TARGET_NS):
    print(f"Norm stats at {FINAL_TARGET_NS}")

for temp_dir in [temp_data, temp_ns]:
    if os.path.exists(temp_dir): shutil.rmtree(temp_dir)

OPENPI_DIR = WORK_DIR
DATASET_CACHE = TARGET_DATA_DIR
NORMSTATS_DIR = FINAL_TARGET_NS

print("Data setup complete")

# STEP 4: Patch Libraries untuk Kompatibilitas

Melakukan patch pada beberapa library (OpenPI, LeRobot, Transformers) untuk memperbaiki issues kompatibilitas dengan versi dependencies yang digunakan.

In [ ]:
def patch_file(filepath, replacements):
    if not os.path.exists(filepath):
        return 0
    
    with open(filepath, 'r') as f:
        content = f.read()
    
    patched = 0
    for old, new in replacements:
        if old in content:
            content = content.replace(old, new)
            patched += 1
    
    if patched > 0:
        with open(filepath, 'w') as f:
            f.write(content)
    
    return patched

print("Patching OpenPI download.py...")
openpi_download = '/kaggle/working/openpi/src/openpi/shared/download.py'
r = patch_file(openpi_download, [('datetime.UTC', 'datetime.timezone.utc')])
print(f"Applied {r} patches")

print("Patching LeRobot dataset.py...")
lerobot_path = os.path.dirname(lerobot.__file__)
lerobot_file = os.path.join(lerobot_path, 'common/datasets/lerobot_dataset.py')
patches = [
    ('torch.stack(self.hf_dataset["timestamp"])', 'torch.stack(list(self.hf_dataset["timestamp"]))'),
    ('torch.stack(self.hf_dataset["episode_index"])', 'torch.stack(list(self.hf_dataset["episode_index"]))'),
    ('torch.stack(self.hf_dataset["index"])', 'torch.stack(list(self.hf_dataset["index"]))'),
    ('torch.stack(self.hf_dataset["task_index"])', 'torch.stack(list(self.hf_dataset["task_index"]))'),
]
r = patch_file(lerobot_file, patches)
print(f"Applied {r} patches")

print("Patching Transformers with OpenPI models...")
trans_path = os.path.dirname(transformers.__file__)
patch_src = '/kaggle/working/openpi/src/openpi/models_pytorch/transformers_replace'
if os.path.exists(patch_src):
    def copy_recursive(src, dst):
        for item in os.listdir(src):
            s, d = os.path.join(src, item), os.path.join(dst, item)
            if os.path.isdir(s):
                if not os.path.exists(d): os.makedirs(d)
                copy_recursive(s, d)
            else:
                shutil.copy2(s, d)
    import shutil
    copy_recursive(patch_src, trans_path)
    print("Models injected")

print("\nVerification:")
print(f"Numpy: {__import__('numpy').__version__}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: Not found")

# STEP 5: Konfigurasi Training Parameters

Mengatur hyperparameter training: jumlah steps, batch size, save interval, dan settings lainnya yang disesuaikan dengan kapasitas GPU.

In [ ]:
CONFIG_NAME = 'pi0_drone_lite'
EXP_NAME = 'drone_gimbal_v1'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'

NUM_STEPS = 3000
BATCH_SIZE = 1
SAVE_INTERVAL = 500
LOG_INTERVAL = 10
MIXED_PRECISION = 'bfloat16'
USE_WANDB = False

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Config:         {CONFIG_NAME}")
print(f"Experiment:     {EXP_NAME}")
print(f"Total Steps:    {NUM_STEPS:,}")
print(f"Batch Size:     {BATCH_SIZE}")
print(f"Save Interval:  {SAVE_INTERVAL} steps")
print(f"Precision:      {MIXED_PRECISION}")
print(f"Checkpoint Dir: {CHECKPOINT_DIR}")
print("Training config ready")

# STEP 6: Pre-Download Assets & Install LoRA

Download PaliGemma tokenizer dan menginstall PEFT library untuk mendukung Low-Rank Adaptation (LoRA) pada model.

In [ ]:
print("Pre-downloading PaliGemma Tokenizer...")
cache_dir = '/root/.cache/openpi/big_vision'
os.makedirs(cache_dir, exist_ok=True)
target_path = os.path.join(cache_dir, "paligemma_tokenizer.model")

if not os.path.exists(target_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/big_vision/paligemma_tokenizer.model",
        target_path
    )
    file_size_mb = os.path.getsize(target_path) / (1024*1024)
    print(f"Downloaded ({file_size_mb:.1f} MB)")
else:
    file_size_mb = os.path.getsize(target_path) / (1024 * 1024)
    print(f"Tokenizer available ({file_size_mb:.2f} MB)")

print("Installing PEFT...")
subprocess.run(["pip", "install", "peft", "-q"], capture_output=True)
print("PEFT installed")
print("Pre-download setup complete")

# STEP 7: Inject LoRA ke Training Script

Memodifikasi training script untuk mengintegrasikan LoRA configuration, sehingga training hanya update parameter low-rank (lebih efisien).

In [ ]:
script_path = '/kaggle/working/openpi/scripts/train_pytorch.py'

subprocess.run(["git", "checkout", "scripts/train_pytorch.py"], cwd="/kaggle/working/openpi", capture_output=True)

with open(script_path, 'r') as f:
    content = f.read()

lora_code = '''
from peft import get_peft_model, LoraConfig
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", bias="none")
model = get_peft_model(model, lora_config)
print(f"LoRA injected | Trainable: {model.get_nb_trainable_parameters()/1e6:.1f}M params")
'''

match = re.search(r'^([ \t]*)model\.train\(\)', content, flags=re.MULTILINE)
indent = match.group(1)
indented = "\n".join(indent + line for line in lora_code.split('\n'))
new_content = content[:match.start()] + indented + "\n\n" + content[match.start():]

with open(script_path, 'w') as f:
    f.write(new_content)

print("LoRA injected into training script")

# STEP 8: Setup Resume Training (OPSIONAL)

Mempersiapkan mekanisme untuk melanjutkan training dari checkpoint terakhir jika ada. Jalankan cell ini hanya jika ingin resume dari checkpoint sebelumnya.

In [ ]:
print("Note: Run this cell only if resuming from a previous checkpoint\n")

target_dir = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
script_path = '/kaggle/working/openpi/scripts/train_pytorch.py'

print("Searching for checkpoint...")
found_dir = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'model.safetensors' in files:
        found_dir = root
        break

if found_dir:
    os.makedirs(target_dir, exist_ok=True)
    for item in os.listdir(found_dir):
        src = os.path.join(found_dir, item)
        dst = os.path.join(target_dir, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print(f"Checkpoint restored to {target_dir}")
else:
    print("No checkpoint found - will start from scratch")

print("Resume setup complete")

# STEP 9: Mulai Training

Menjalankan main training loop. Proses ini akan iterasi melalui dataset sesuai jumlah steps yang dikonfigurasi dan menyimpan checkpoint secara berkala.

In [ ]:
torch.cuda.empty_cache()

env = os.environ.copy()
env['PYTHONPATH'] = "/kaggle/working/openpi/src:/kaggle/working/openpi/packages/openpi-client/src"
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_OFFLINE'] = '1'
env['JAX_PLATFORMS'] = 'cpu'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

cmd = [
    sys.executable, '/kaggle/working/openpi/scripts/train_pytorch.py', CONFIG_NAME,
    f'--exp_name={EXP_NAME}', f'--batch_size={BATCH_SIZE}', f'--num_train_steps={NUM_STEPS}',
    f'--save_interval={SAVE_INTERVAL}', f'--log_interval={LOG_INTERVAL}',
]
if not USE_WANDB:
    cmd.append('--no-wandb-enabled')

print(f"Config: {CONFIG_NAME} | Steps: {NUM_STEPS} | Batch: {BATCH_SIZE}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram:.1f}GB VRAM)")

print("Training started...\n")

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, 
                          stderr=subprocess.STDOUT, text=True, bufsize=1, cwd='/kaggle/working')

for line in process.stdout:
    if "Unable to initialize backend" not in line:
        print(line, end='')
        
exit_code = process.wait()

print("\n" + "="*60)
if process.returncode == 0:
    print("Training completed")
else:
    print(f"Training exited with code {process.returncode}")
print("="*60)

# STEP 10: Monitor Checkpoint Progress

Menampilkan daftar file checkpoint yang telah tersimpan. Jalankan cell ini berkali-kali untuk memantau progress training.

In [ ]:
target_dir = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
files = os.listdir(target_dir)

total_size = 0
print("Checkpoint files:\n")

for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(target_dir, f)) / (1024**2)
    total_size += size_mb
    mod_time = time.ctime(os.path.getmtime(os.path.join(target_dir, f)))
    print(f"{f:30} | {size_mb:7.1f}MB | {mod_time}")

print(f"\nTotal: {total_size:.1f}MB")
print("Checkpoint complete")

# STEP 11: Download Latest Checkpoint

Mengompres checkpoint terbaru menjadi file ZIP dan menyiapkan link download. Gunakan link ini untuk download model yang sudah dilatih.

In [ ]:
folder = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
zip_path = '/kaggle/working/checkpoint_LATEST'

shutil.make_archive(zip_path, 'zip', folder)
zip_file = f'{zip_path}.zip'
size_mb = os.path.getsize(zip_file) / (1024**2)

print(f"Compression complete ({size_mb:.1f}MB)\n")
print("="*60)
print("Download link:\n")
display(FileLink(zip_file))
print("\n" + "="*60)

# STEP 12: Test Inference (OPSIONAL)

Menguji model yang sudah dilatih dengan memberikan sample image dan prompt untuk melihat prediksi action dari model.

In [ ]:
test_dir = '/kaggle/working/test_checkpoint'
dirty_path = os.path.join(test_dir, "model.safetensors")
clean_path = os.path.join(test_dir, "model_clean.safetensors")

print("Restoring checkpoint...")
found_dir = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'model.safetensors' in files:
        found_dir = root
        break

if found_dir and not os.path.exists(test_dir):
    os.makedirs(test_dir, exist_ok=True)
    for item in os.listdir(found_dir):
        src = os.path.join(found_dir, item)
        dst = os.path.join(test_dir, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print("Checkpoint ready")
elif os.path.exists(test_dir):
    print("Already restored")
else:
    raise FileNotFoundError("Checkpoint not found")

print("Cleaning LoRA weights...")
if not os.path.exists(clean_path):
    state_dict = load_file(dirty_path)
    clean_dict = {k.replace("base_model.model.", ""): v for k, v in state_dict.items()}
    save_file(clean_dict, clean_path)
print("Weights cleaned")

print("Loading test image...")
ds = load_dataset("rafli/drone_roblox", split="train")
img_col = 'image' if 'image' in ds.column_names else ds.column_names[0]
img = ds[0][img_col]

if img.mode != 'RGB': 
    img = img.convert('RGB')
print("Image ready")

PROMPT = "terbang maju menuju kubus merah"
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(img)
ax.set_title(f"Prompt: '{PROMPT}'", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

print("Running inference...")
os.rename(dirty_path, os.path.join(test_dir, "model_dirty.bak"))
os.rename(clean_path, dirty_path)

policy = policy_factory.create_policy(
    config_name="pi0_drone_lite",
    checkpoint_dir=test_dir
)

action = policy({"image": img, "prompt": PROMPT})

print("Inference successful")
print(f"Predicted action:\n{action}\n")

if os.path.exists(os.path.join(test_dir, "model_dirty.bak")):
    if os.path.exists(dirty_path):
        os.rename(dirty_path, clean_path)
    os.rename(os.path.join(test_dir, "model_dirty.bak"), dirty_path)

print("Inference test complete")